# Reproducing Liu et al. (2026) from dandiset 001750

**Paper.** Liu R-F, Huang M, Shen Y, Shao M, Jing J et al. (2026). An atlas of primate insular
cortex reveals a signal-processing strategy in von Economo neurons. *Nature Cell Biology* 28,
1742-1757. [doi:10.1038/s41556-026-02009-4](https://doi.org/10.1038/s41556-026-02009-4). The
claims file was written from the version of record (Europe PMC, PMC13457102): main text, legends,
Methods; this attempt also had the Supplementary Information PDF, which says nothing about the Na+
recordings.

**Dandiset.** [DANDI:001750](https://dandiarchive.org/dandiset/001750/0.260825.0222), version
0.260825.0222. The paper cites no version. This is the only published version, and the most
recent; the draft has the same 35 assets and size. It is pinned for that reason.

**What the dandiset holds.** 35 NWB files, one per somatic outside-out patch, from one macaque
(Mfa-221206), sessions 221206AS (17 files) and 221206HS (18 files): voltage-clamp currents to
5 mV steps from -80 mV, 1 to 10 repetitions per patch. No cell type and no patch capacitance.

**Outside inputs.** Cell types come from the paper's code table
`ephysio/data_ephy/rawdata_ephysio/metadata_outSideOutSodiumCurrentRec.csv` in
github.com/RFLiu2021/AIC_proj at commit 44cd8a6 (claims `with_labels`). Two claims on other data
(`elsewhere`) are tested on MetaboLights MTBLS13927, metadata revision MTBLS13927_01, and one on
`ephysio/data_ephy/sumData_StimWM.xlsx` of the same repository commit.

| Claim | Data | Tested here |
|---|---|---|
| na-cell-table-link | with labels | tested |
| fig4d-sodium-current-lower | dataset (labels from the link) | tested |
| model-na-ratio | dataset | tested (depends on fig4d) |
| na-patch-capacitance | dataset | blocked, no capacitance in the files |
| fig8a-metabolite-gradient, fig8de-glycolytic-bias | elsewhere (MetaboLights) | tested |
| fig4j-eepsp-amplitude | elsewhere (code repository table) | tested |
| fig3c-ais-distance-length | elsewhere (code repository table) | blocked, no slice or animal labels |
| all other claims (Patch-seq, connectivity, PSCs, scRNA-seq, morphology, models, eEPSP latency) | elsewhere | blocked, see the record |

In [ ]:
import io
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.stats.anova import AnovaRM, anova_lm
from statsmodels.stats.oneway import anova_oneway

import nacurrent as nc

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
assets = nc.list_assets()
patches = nc.load_all()
table = nc.cell_table()
print(f"streamed {len(patches)} files ({assets['size'].sum() / 1e9:.2f} GB) of dandiset {nc.DANDISET} "
      f"version {nc.VERSION} in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
info = pd.DataFrame([{
    "file": p["file"][24:], "session": p["meta"]["session_id"], "subject": p["meta"]["subject_id"],
    "cell_id": p["meta"]["cell_id"], "location": p["meta"]["location"], "reps": p["nrep"],
    "sweeps": len(p["sweeps"]), "samples": int(p["info"].n.iloc[0]), "rate_Hz": round(p["rate"]),
    "onset_ms": p["onset"] / p["rate"] * 1e3, "step_ms": p["step_ms"],
    "levels_mV": f"{p['levels'].min():.0f} to {p['levels'].max():.0f}",
    "unit": "/".join(sorted(set(p["info"].unit))), "stim_unit": "/".join(sorted(set(p["info"].stim_unit))),
    "series_fields": "/".join(sorted(set(p["info"].fields))),
    "holding_pA": round(float(np.median(p["mean"][:, :p["onset"] - 100])), 1),
    "noise_sd_pA": round(p["rep_sd"], 2)} for p in patches])
print("Files per session:", info.session.value_counts().to_dict(), "; subjects:", info.subject.unique().tolist())
print("Sampling rates (Hz):", sorted(info.rate_Hz.unique()), "; samples per sweep:", sorted(info.samples.unique()))
print("Step onset (ms):", sorted(info.onset_ms.unique()), "; step duration (ms):", sorted(info.step_ms.unique()))
print("Sweeps per file:", info.sweeps.value_counts().to_dict(), "; repetitions per file:", info.reps.value_counts().sort_index().to_dict())
print("Units:", info.unit.unique().tolist(), info.stim_unit.unique().tolist())
print("Datasets in each VoltageClampSeries:", info.series_fields.unique().tolist())
print("General groups of the first file:", patches[0]["meta"]["keys_general"])
print("session_start_time values:", sorted({p["meta"]["session_start"] for p in patches}),
      "; session ids:", sorted({p["meta"]["session_id"] for p in patches}))
# Step levels: sweep s should step to -80 + 5 (s - 1) mV.
dev = max(np.abs(p["levels"] - (-80 + 5 * (p["sweeps"] - 1))).max() for p in patches)
print("Largest deviation of a sweep's step level from -80 + 5 (sweep - 1) mV:", round(float(dev), 3))
print("Command before the step, every file and sweep (mV):",
      sorted({round(float(np.median(p["command"][0][:, :p["onset"] - 1])), 1) for p in patches}))
# Recordings deposited twice: identical traces.
dup = info.assign(digest=[p["digest"] for p in patches]).digest.duplicated(keep=False)
print("Files whose repetition-1 traces are identical to another file's:", int(dup.sum()))
print("\n", info.drop(columns=["series_fields", "unit", "stim_unit", "subject", "location"]).to_string(index=False))
print("\nCurrent range over all files and samples (pA):",
      round(min(p["mean"].min() for p in patches), 1), "to", round(max(p["mean"].max() for p in patches), 1))

All 35 files are from one subject, sampled at 100 kHz (the Methods' rate), with 6500 samples
(65 ms) per sweep and the step from 20 ms lasting 25 ms (the Methods' 25 ms). Sweep s steps to
-80 + 5 (s - 1) mV, so the Methods' -80 to 0 mV are sweeps 1 to 17; files hold 28 sweeps (to
+55 mV) or 34 (to +85 mV). Sweep 1 is the -80 mV "step", with no change of command. The files carry
no capacitance or series resistance compensation values (only `data`, `electrode`, `gain`,
`starting_time` in each series), no cell type, and no intracellular recording tables. The
holding current before the step is far from zero in every file (printed `holding_pA`), so the
traces, though leak subtracted online by the Methods, keep an offset. No two files hold the same
traces. The `starting_time` of the series (0, 1, 2 or 3 s) does not order the sweeps and is not
used. Every file gives the same session start, 2018-01-01 01:00, while the session ids
(221206AS, 221206HS) and the code table's cell IDs (20221206...) date the recordings to
6 December 2022; the start time is a placeholder.

In [ ]:
# The response follows the command: mean current 0.2 to 2 ms after the step onset, minus the 1 ms
# before it, at the steps where Na+ currents peak (-40 to 0 mV) against sweep 1 (no step), and the
# same contrast with the window moved to 10 to 8 ms before the onset (labels shifted in time).
rows = []
for p in patches:
    m, on, r = p["mean"], p["onset"], p["rate"]
    def resp(j, shift=0):
        a, b = on + shift, on + shift
        return m[j, a + int(0.0002 * r):a + int(0.002 * r)].mean() - m[j, b - int(0.001 * r):b].mean()
    act = np.flatnonzero((p["levels"] >= -40) & (p["levels"] <= 0))
    rows.append({"file": p["file"][24:], "step": np.mean([resp(j) for j in act]), "no_step": resp(0),
                 "step_shifted": np.mean([resp(j, -int(0.010 * r)) for j in act])})
resp = pd.DataFrame(rows)
print(resp.describe().loc[["mean", "50%", "min", "max"]].round(2).to_string())
print("Files where the step response is more inward than sweep 1:", int((resp.step < resp.no_step).sum()), "of", len(resp))
print("Wilcoxon, step against no step: p = %.2g; shifted window against no step: p = %.2g" % (
    stats.wilcoxon(resp.step, resp.no_step).pvalue, stats.wilcoxon(resp.step_shifted, resp.no_step).pvalue))

The inward current appears after the command step and not in the same window shifted 10 ms
earlier, so the current and the command share one clock in every file, as far as this contrast
can tell.

## na-cell-table-link

> The dandiset's recordings are the 35 outside-out patches of the paper's code table (14 VEN and
> 21 PC), and the maximal Na+ current computed from each recording agrees with the value the table
> gives for it.

Matched by NWB file name. The maximal inward current is computed under each reading of
fig4d-sodium-current-lower: baseline (a1) as recorded or (a2) minus the mean of the first 100
samples of a cut starting 2 ms before onset; steps (b1) -80 to 0 mV or (b2) all; window (c1)
0 to 20 ms or (c2) 0 to 25 ms after onset. Repetitions are averaged first.

In [ ]:
meas = nc.measure_all(patches)
link = table.merge(meas, left_on="fileName_nwb", right_on="file", how="outer", indicator=True)
print("Table rows:", len(table), "; files:", len(meas), "; matched:", int((link._merge == "both").sum()))
print("Cell types in the table:", table.cellType.value_counts().to_dict())
link["file_cell_id"] = link.file.map({p["file"]: p["meta"]["cell_id"] for p in patches})
# Table cell_ID 20221206AS2c6 against file cell_id Cell_AS006: session letters and cell number.
key = lambda s: (s.str.extract(r"(AS|HS)")[0] + s.str.extract(r"c(\d+)$")[0].astype(int).astype(str))
link["id_agrees"] = key(link.cell_ID) == (link.file_cell_id.str.extract(r"(AS|HS)")[0]
                                          + link.file_cell_id.str.extract(r"(\d+)$")[0].astype(int).astype(str))
print("Table cell_ID agrees with the file's cell_id (session and cell number):", int(link.id_agrees.sum()), "of", len(link))
print("Cell type in the dataset: none (no field holds it), so no type can disagree.")
link["table_pA"] = link.max_Na_current.abs()
cols = ["".join(r) for r in nc.READINGS]
agree = pd.Series({c: int((np.abs(link[c] - link.table_pA) <= 0.1 * link.table_pA).sum()) for c in cols})
ratio = pd.Series({c: float(np.median(link[c] / link.table_pA)) for c in cols})
rho = pd.Series({c: stats.spearmanr(link[c], link.table_pA)[0] for c in cols})
print("\nPer reading: cells within 10% of the table (of 35), median computed/table, Spearman rho")
print(pd.DataFrame({"within_10pct": agree, "median_ratio": ratio.round(2), "spearman": rho.round(3)}).to_string())
print("\nPer cell (pA):")
print(link[["cell_ID", "cellType", "table_pA"] + cols + ["level_mV", "latency_ms"]].round(1).to_string(index=False))

The criterion's readings take the most negative single sample of the averaged trace. Since noise
pushes that minimum down, the next cell runs a variant that is not a reading of the paper and
decides nothing: a moving average over n samples before the peak is taken. It was added after the
readings were run, to see what measure the table's values could come from.

In [ ]:
rows = []
for n in [1, 25, 50, 75, 100, 150]:
    v = nc.measure_all(patches, smooth=n).set_index("file").loc[link.file]
    for c in ["a2b1c1", "a2b1c2"]:
        x = v[c].values
        rows.append({"smooth_samples": n, "smooth_ms": n / 100, "reading": c,
                     "within_10pct": int((np.abs(x - link.table_pA) <= 0.1 * link.table_pA).sum()),
                     "median_ratio": round(float(np.median(x / link.table_pA)), 3),
                     "spearman": round(stats.spearmanr(x, link.table_pA)[0], 3)})
smooth_tab = pd.DataFrame(rows)
print(smooth_tab.to_string(index=False))

**Result: partially reproduced.** All 35 files match the 35 table rows by file name, and each
table cell ID agrees with the file's cell ID (35 of 35). The dataset holds no cell type, so none
disagrees. Under the criterion's readings at most 2 of 35 maxima are within 10% of the table,
below the 28 the criterion requires. The as-recorded readings (a1) are a median 4.15 to 4.61 times
the table values, as the traces keep a holding offset. The baseline-subtracted readings (a2) are a
median 1.32 to 1.40 times the table values, with Spearman 0.47 to 0.68; a2b1c1 agrees best. In the
non-deciding variant that smooths the trace over 0.75 ms before taking the peak, 26 of 35 agree
within 10% (median ratio 1.006, Spearman 0.982). So the table's values are consistent with a
filtered peak of these recordings. That variant was found after looking at the data.

## fig4d-sodium-current-lower

> Somatic Na+ currents recorded in outside-out patches are significantly lower in VENs (n = 14)
> than in L5 PCs (n = 21).

Cell types from the code table (link above). Two-tailed Wilcoxon rank-sum test (Mann-Whitney U)
under each of the eight readings.

In [ ]:
ven, pc = link.cellType == "VEN", link.cellType == "PC"
def compare(x):
    return {"VEN_n": int(ven.sum()), "PC_n": int(pc.sum()), "VEN_median": np.median(x[ven]), "PC_median": np.median(x[pc]),
            "ranksum_p": stats.mannwhitneyu(x[ven], x[pc], alternative="two-sided").pvalue,
            "ttest_p": stats.ttest_ind(x[ven], x[pc]).pvalue,
            "VEN_lower": bool(np.median(x[ven]) < np.median(x[pc]))}
res = pd.DataFrame({c: compare(link[c].values) for c in cols}).T
res.loc["code table (paper's values)"] = compare(link.table_pA.values)
sm75 = nc.measure_all(patches, smooth=75).set_index("file").loc[link.file]
res.loc["variant: a2b1c1 smoothed 0.75 ms"] = compare(sm75["a2b1c1"].values)
print(res.to_string(float_format=lambda v: f"{v:.4g}"))
print("\nReadings with VEN lower and p < 0.05:", [c for c in cols if res.loc[c, "VEN_lower"] and res.loc[c, "ranksum_p"] < 0.05])
print("\nBy session, reading a2b1c1 (VEN median, PC median, n):")
for s, g in link.assign(session=link.cell_ID.str[8:10]).groupby("session"):
    print(s, round(g[g.cellType == "VEN"].a2b1c1.median(), 1), round(g[g.cellType == "PC"].a2b1c1.median(), 1),
          int((g.cellType == "VEN").sum()), int((g.cellType == "PC").sum()))
print("Current density cannot be computed: no patch capacitance in the files (see na-patch-capacitance).")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for i, (c, lab) in enumerate([("a2b1c1", "reading a2 b1 c1"), ("table_pA", "code table")]):
    ax[i].boxplot([link[c][ven], link[c][pc]], tick_labels=["VEN", "PC"])
    ax[i].scatter(np.r_[np.ones(ven.sum()), 2 * np.ones(pc.sum())], np.r_[link[c][ven], link[c][pc]], s=12, alpha=.6)
    ax[i].set_ylabel("max inward current (pA)"); ax[i].set_title(lab)
plt.tight_layout(); plt.show()
print("Plotted ranges (pA): a2b1c1", round(link.a2b1c1.min(), 1), "to", round(link.a2b1c1.max(), 1),
      "; table", round(link.table_pA.min(), 1), "to", round(link.table_pA.max(), 1), "; no point outside the axes")

**Result: blocked (methods underspecified).** The counts are 14 VEN and 21 PC, as the paper
states, and the VEN median is lower under all eight readings. But the readings disagree on
significance: a1b1c1 (p = 0.045) and a1b1c2 (p = 0.025) find it, and the six others do not
(p = 0.077 to 0.29), so by the criterion the claim is blocked. Under the code's example reading
(a2 b1 c1) the medians are 30.4 against 38.1 pA (p = 0.10). By session under that reading, VENs
are lower in session HS (15.9 against 40.3 pA) but not in AS (40.7 against 36.2 pA). The code
table's own values give 14.9 against 29.3 pA (p = 0.001), and the non-deciding smoothed variant
gives 16.0 against 29.9 pA (p = 0.0008). The outcome turns on how the peak is measured, which
the paper does not say.

## model-na-ratio

> The VEN model's somatodendritic Na+ conductance (55 pS/um2, against 80 pS/um2 in the PC model)
> was set from the recorded ratio of peak Na+ current in VENs to PCs, which puts that ratio near
> 55/80, about 0.69.

Expected 0.6875 with tolerance 0.069 (0.619 to 0.756), as mean over mean (i) or median over median
(ii), under each reading of the measure.

In [ ]:
lo, hi = 0.6875 - 0.069, 0.6875 + 0.069
rows = []
for c in cols + ["table_pA"]:
    x = link[c].values
    r_mean, r_med = x[ven].mean() / x[pc].mean(), np.median(x[ven]) / np.median(x[pc])
    rows.append({"measure": c, "mean_ratio": r_mean, "median_ratio": r_med,
                 "mean_in": lo <= r_mean <= hi, "median_in": lo <= r_med <= hi})
x = sm75["a2b1c1"].values
rows.append({"measure": "variant smoothed 0.75 ms", "mean_ratio": x[ven].mean() / x[pc].mean(),
             "median_ratio": np.median(x[ven]) / np.median(x[pc])})
ratios = pd.DataFrame(rows)
print(ratios.round(3).to_string(index=False))
print("Readings of the measure with a ratio in tolerance under some summary:",
      [r.measure for r in ratios.itertuples() if r.measure in cols and (r.mean_in or r.median_in)])

**Result: blocked (methods underspecified).** The claim depends on fig4d-sodium-current-lower,
which is blocked. Under the eight readings the ratios printed above are 0.68 to 0.99. A ratio falls
within 0.619 to 0.756 under some summary only for the four as-recorded readings (a1), none of the
baseline-subtracted ones (0.79 to 0.99). The code table's values give 0.58 (mean) and 0.51
(median), and the smoothed variant 0.58 and 0.54; none of these is in tolerance.

## na-patch-capacitance

> The outside-out patches have a capacitance of about 50 fF, a membrane area of about 5 um2 at
> 1 uF/cm2, and similar membrane areas were sampled from both cell types.

In [ ]:
fields = sorted(set("/".join(sorted(set(p["info"].fields))) for p in patches))
print("Datasets in the VoltageClampSeries of all files:", fields)
cap_words = ("capacitance", "resistance", "whole_cell", "compensation")
print("Any series field naming capacitance or compensation:", any(w in f for f in fields for w in cap_words))
meta = nc.asset_metadata(assets.asset_id.iloc[0])
print("Asset metadata keys:", sorted(meta))
print("Any asset metadata value naming capacitance:", "capacitance" in str(meta).lower())

**Result: blocked (data missing).** No file holds a capacitance, compensation or series
resistance value, and the asset metadata names none. The criterion blocks the claim in that case.
The traces are leak subtracted online (Methods), so the capacitive transients from which a
capacitance might be estimated are not there either.

## fig8a-metabolite-gradient (elsewhere: MetaboLights MTBLS13927)

> Of 581 metabolites, 228 differ between AIC and V1 and only 108 between AIC and parietal cortex.

Welch's ANOVA followed by Games-Howell at P < 0.05 per metabolite, 6 AIC, 6 PrC and 7 V1 samples
(Fig. 8 legend). Values as deposited in the metabolite assignment file; the Methods name no
transform, so log2 values are a variant. "Followed by" is read as the Games-Howell pair counted
when the Welch ANOVA gives P < 0.05; the pair test alone is shown as a variant.

In [ ]:
MTBLS = "https://ftp.ebi.ac.uk/pub/databases/metabolights/studies/public/MTBLS13927/METADATA_REVISIONS/MTBLS13927_01/"
maf = pd.read_csv(io.StringIO(requests.get(MTBLS + "m_MTBLS13927_LC-MS_alternating_hilic_metabolite_profiling_v2_maf.tsv", timeout=120).text), sep="\t")
samples = pd.read_csv(io.StringIO(requests.get(MTBLS + "s_MTBLS13927.txt", timeout=120).text), sep="\t")
scols = [c for c in maf.columns if c[:3] in ("A1C", "AIC", "PrC", "V1-")]
region = {c: "AIC" if c[:3] in ("A1C", "AIC") else ("PrC" if c.startswith("PrC") else "V1") for c in scols}
print("Metabolites:", len(maf), "; sample columns:", len(scols), pd.Series(region).value_counts().to_dict())
print("Sample table names:", samples["Source Name"].tolist())
print("Column 'A1C-1-Q' is read as AIC-1-Q (the sample table spells it so too; the other AIC samples are AIC-).")
X = maf[scols].apply(pd.to_numeric, errors="coerce")
print("Missing values:", int(X.isna().sum().sum()), "; zeros:", int((X == 0).sum().sum()), "; range", float(X.min().min()), "to", float(X.max().max()))
print("Metabolites without a name:", int(maf.metabolite_identification.isna().sum()),
      "; duplicate names:", int(maf.metabolite_identification.duplicated().sum()))

def games_howell(a, b):
    na, nb, va, vb = len(a), len(b), a.var(ddof=1), b.var(ddof=1)
    se = np.sqrt(va / na + vb / nb)
    if se == 0:
        return np.nan
    df = (va / na + vb / nb) ** 2 / ((va / na) ** 2 / (na - 1) + (vb / nb) ** 2 / (nb - 1))
    q = abs(a.mean() - b.mean()) / se * np.sqrt(2)
    return stats.studentized_range.sf(q, 3, df)

def metab_tests(M):
    out = []
    for i in range(len(M)):
        g = {r: M.iloc[i][[c for c in scols if region[c] == r]].values.astype(float) for r in ("AIC", "PrC", "V1")}
        w = anova_oneway([g["AIC"], g["PrC"], g["V1"]], use_var="unequal").pvalue
        out.append({"welch_p": w, "gh_AIC_V1": games_howell(g["AIC"], g["V1"]), "gh_AIC_PrC": games_howell(g["AIC"], g["PrC"]),
                    "AIC": g["AIC"].mean(), "PrC": g["PrC"].mean(), "V1": g["V1"].mean()})
    return pd.DataFrame(out, index=maf.metabolite_identification)

tests = {"as deposited": metab_tests(X), "log2 (variant)": metab_tests(np.log2(X.clip(lower=1e-9)))}
rows = []
for name, t in tests.items():
    gate = t.welch_p < 0.05
    rows.append({"values": name, "welch_p<0.05": int(gate.sum()),
                 "AIC-V1 (Welch then GH)": int((gate & (t.gh_AIC_V1 < 0.05)).sum()),
                 "AIC-PrC (Welch then GH)": int((gate & (t.gh_AIC_PrC < 0.05)).sum()),
                 "AIC-V1 (GH alone)": int((t.gh_AIC_V1 < 0.05).sum()),
                 "AIC-PrC (GH alone)": int((t.gh_AIC_PrC < 0.05).sum())})
print(pd.DataFrame(rows).to_string(index=False))
print("Expected 228 (205.2 to 250.8) AIC-V1 and 108 (97.2 to 118.8) AIC-PrC, of 581 metabolites.")

**Result: not reproduced.** Welch's ANOVA followed by Games-Howell gives 175 metabolites that
differ between AIC and V1 (expected 205.2 to 250.8) and 84 between AIC and PrC (expected 97.2 to
118.8). The Games-Howell test alone gives 179 and 85, and on log2 values 190 and 95 (194 and 98
without the ANOVA gate), all outside tolerance. The direction holds, with more differences
against V1 than against PrC under every variant. The deposited table holds 558 metabolites, not
the 581 the Results report, which may account for part of the shortfall.

## fig8de-glycolytic-bias (elsewhere: MetaboLights MTBLS13927)

> Compared with V1, AIC shows elevated citrate, reduced 2-ketobutyrate and increased glutamate,
> read as a glycolytic bias.

Welch's ANOVA followed by Games-Howell, AIC against V1. The metabolites are found by name in the
deposited table; every candidate name is printed.

In [ ]:
names = maf.metabolite_identification.fillna("")
cand = {"citrate": names.str.contains(r"citr(?:ate|ic)", case=False) & ~names.str.contains("iso|methyl", case=False),
        "isocitrate (shown only)": names.str.contains("isocitr", case=False),
        "2-ketobutyrate": names.str.contains(r"2-ketobutyr|2-oxobutan|alpha-ketobutyr", case=False),
        "glutamate": names.str.contains(r"^(?:L-)?glutam(?:ate|ic acid)", case=False)}
t = tests["as deposited"]; tl = tests["log2 (variant)"]
rows = []
for k, m in cand.items():
    for i in np.flatnonzero(m.values):
        rows.append({"target": k, "name": names.iloc[i], "AIC_mean": t.AIC.iloc[i], "V1_mean": t.V1.iloc[i],
                     "AIC/V1": t.AIC.iloc[i] / t.V1.iloc[i], "welch_p": t.welch_p.iloc[i], "gh_AIC_V1": t.gh_AIC_V1.iloc[i],
                     "gh_AIC_V1_log2": tl.gh_AIC_V1.iloc[i]})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print("Names containing 'glutam' (all):", sorted(n for n in names if "glutam" in n.lower()))
print("Names containing 'citr' or 'butyr' (all):", sorted(n for n in names if "citr" in n.lower() or "butyr" in n.lower()))

**Result: partially reproduced.** Citric acid is higher in AIC than in V1 (ratio 1.37,
Games-Howell p = 0.00046), and L-glutamic acid too (ratio 1.35, p = 0.0001). 2-Ketobutyric acid is
lower (ratio 0.37) but not significantly (Games-Howell p = 0.18, Welch's ANOVA p = 0.17; on log2
values p = 0.086). The entry named "Glutamic acid glutamate" is higher (1.18) without significance
(p = 0.15); it is taken as a second glutamate feature, not as the paper's glutamate.

## fig4j-eepsp-amplitude (elsewhere: code repository table)

> eEPSP amplitudes are higher in VENs than in adjacent PCs, with effects of cell type and
> stimulation level in an ANOVA.

From `ephysio/data_ephy/sumData_StimWM.xlsx` (AIC_proj at commit 44cd8a6), sheet
`re-orgnized_data`, the "copied data" block: per VEN-PC pair, the eEPSP amplitudes of the PC and
the VEN at three stimulation levels. Readings (a) ordinary two-way ANOVA (cell type and
stimulation level) and (b) repeated-measures ANOVA with the pair as subject.

In [ ]:
raw = pd.read_excel(io.BytesIO(nc.repo_file("ephysio/data_ephy/sumData_StimWM.xlsx")), sheet_name="re-orgnized_data", header=None)
block = raw.iloc[1:, 0:4]
block.columns = ["pair", "stim", "PC", "VEN"]
rows, pair, level = [], None, 0
for _, r in block.iterrows():
    if pd.isna(r.PC) and pd.isna(r.VEN):
        pair, level = None, 0
        continue
    if pair is None:
        pair = r.pair if isinstance(r.pair, str) else f"block{len(set(x['pair'] for x in rows)) + 1}"
    level += 1
    rows.append({"pair": pair, "level": level, "stim": r.stim, "PC": float(r.PC), "VEN": float(r.VEN)})
ep = pd.DataFrame(rows)
print("Pairs:", ep.pair.nunique(), "; rows per pair:", ep.groupby("pair").size().value_counts().to_dict())
print("Stimulation values given:", sorted(ep.stim.dropna().unique().tolist()), "; rows without a value:", int(ep.stim.isna().sum()))
print("Amplitude range (unit not stated in the sheet): PC", ep.PC.min(), "to", ep.PC.max(), "; VEN", ep.VEN.min(), "to", ep.VEN.max())
long = ep.melt(id_vars=["pair", "level"], value_vars=["PC", "VEN"], var_name="cell", value_name="amp")
print("\nMean amplitude by level and cell type:")
print(long.pivot_table(index="level", columns="cell", values="amp", aggfunc=["mean", "median", "count"]).to_string())
fit = smf.ols("amp ~ C(cell) * C(level)", data=long).fit()
print("\n(a) two-way ANOVA (type II):")
print(anova_lm(fit, typ=2).to_string(float_format=lambda v: f"{v:.4g}"))
full = long.groupby("pair").filter(lambda g: len(g) == 6)
print(f"\n(b) repeated-measures ANOVA, {full.pair.nunique()} pairs with all three levels:")
print(AnovaRM(full, "amp", "pair", within=["cell", "level"]).fit().anova_table.to_string(float_format=lambda v: f"{v:.4g}"))
d = ep.groupby("pair")[["VEN", "PC"]].mean()
print("\nPairs where the VEN mean amplitude exceeds the PC's:", int((d.VEN > d.PC).sum()), "of", len(d),
      "; Wilcoxon signed-rank p = %.3g" % stats.wilcoxon(d.VEN, d.PC).pvalue)

**Result: reproduced.** In the 26 pairs of the table, the VEN amplitudes are higher at every
stimulation level (mean 0.0067, 0.0133 and 0.0193 against 0.0036, 0.0066 and 0.0128), and in 21 of
26 pairs. The cell-type effect is significant under both readings: the ordinary two-way ANOVA
gives F = 7.62, p = 0.0065, and the repeated-measures ANOVA F = 8.72, p = 0.0068. The paper
reports P = 0.0034. The stimulation level effect has p = 5.3e-05 and 1.6e-08 respectively. The
table gives stimulation values of 0.5, 1 and 2 (unit "mA") where the Methods give 0.05 to 0.2 mA,
and no unit for the amplitudes.

## fig3c-ais-distance-length (elsewhere: code repository table)

> In Pan-Nav stained sections the AIS of VENs lies farther from the soma than that of PCs, and VENs
> have a shorter AIS.

From `morpho/dataMorph/AIScmp_PCvsVEN_Fig5c.xlsx` (AIC_proj at commit 44cd8a6). The criterion's
test is a two-way ANOVA with slice or animal as the second factor (n = 108 VENs and 430 PCs, 23
slices, five animals, Fig. 3c legend).

In [ ]:
ais = pd.read_excel(io.BytesIO(nc.repo_file("morpho/dataMorph/AIScmp_PCvsVEN_Fig5c.xlsx")), header=None)
heads = {}
for j in ais.columns:
    hit = ais[j][ais[j].astype(str).str.contains("AIS_")]
    if len(hit):
        heads[j] = hit.iloc[0]
print("Columns:", heads)
vals = {h: pd.to_numeric(ais[j], errors="coerce").dropna() for j, h in heads.items()}
for h, v in vals.items():
    print(f"{h}: n = {len(v)}, median {v.median():.2f}, range {v.min():.2f} to {v.max():.2f}")
print("Other columns with values:", [j for j in ais.columns if j not in heads and ais[j].notna().sum() > 2])
print("Legend: n = 108 VENs and 430 PCs, 23 slices, five animals; the sheet has no slice or animal column.")
for a, b in [("VEN_AIS_D (um)", "PC_AIS_D (um)"), ("VEN_AIS_L (um)", "PC_AIS_L (um)")]:
    print(f"Description only, rank-sum {a} against {b}: p = {stats.mannwhitneyu(vals[a], vals[b]).pvalue:.3g}")

**Result: blocked (metadata insufficient).** The sheet holds AIS length and distance per cell,
by cell type, with no slice or animal, so neither reading of the two-way ANOVA can be formed. Its
counts (printed above) differ from the legend's 108 VENs and 430 PCs. The rank-sum tests printed
above describe the sheet and decide nothing.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Data | Outcome |
|---|---|---|
| na-cell-table-link | with labels | partially reproduced |
| fig4d-sodium-current-lower | dataset (types from the link) | blocked: methods underspecified |
| model-na-ratio | dataset | blocked: methods underspecified (depends on fig4d) |
| na-patch-capacitance | dataset | blocked: data missing |
| fig8a-metabolite-gradient | elsewhere (MetaboLights) | not reproduced |
| fig8de-glycolytic-bias | elsewhere (MetaboLights) | partially reproduced |
| fig4j-eepsp-amplitude | elsewhere (code repository table) | reproduced |
| fig3c-ais-distance-length | elsewhere (code repository table) | blocked: metadata insufficient |
| 26 other claims on data elsewhere | elsewhere | blocked (see the record) |

**How the analysis differs from the paper's.** The paper does not say how the maximal Na+ current
was measured from the traces; the claims file's eight readings were run, and they disagree on
significance. The code table's values match a peak taken after smoothing the traces, a step the
paper does not describe. Current density could not be computed, since the files hold no
capacitance. The metabolomics counts use the deposited values with no transform, which the
Methods do not name.

**What was found about the dataset.** The 35 files match the paper's code table one to one by
file name and cell number, with 14 VEN and 21 PC by that table, but the files carry no cell type,
so the dataset alone cannot be split into VENs and PCs. No file holds the patch capacitance or
any amplifier compensation value. The traces keep holding currents of -131 to +92 pA, so a
baseline must be subtracted before a peak is taken; one file (HS018) has a positive holding
current and noise near 57 pA. Files hold 28 or 34 sweeps (to +55 or +85 mV) and 1 to 10
repetitions, where the Methods describe steps to 0 mV. The series' starting times (0 to 3 s) carry
no order.